# Instalação de libs

In [0]:
!pip install -q openpyxl xlsxwriter

In [0]:
dbutils.library.restartPython()

# Parâmetros do notebook

In [0]:
# dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("environment", "dev", "Environment")
environment = dbutils.widgets.get("environment")
print("Environment:", environment)

In [0]:
dbutils.widgets.text("id_projeto", "endometriose", "ID Projeto")
id_projeto = dbutils.widgets.get("id_projeto")
print("id_projeto:", id_projeto)

In [0]:
from datetime import datetime

dbutils.widgets.text("data_execucao_modelo", "", "Data Execução Modelo")
data_execucao_modelo = dbutils.widgets.get("data_execucao_modelo")
if data_execucao_modelo == "":
    data_execucao_modelo = datetime.now().strftime("%Y-%m-%d")

print(f"Data Execução Modelo: {data_execucao_modelo}")

In [0]:
if environment in ["hml", "prd"]:
    environment_tbl = ""
else:
    environment_tbl = f"{environment}_"

print("environment_tbl:", environment_tbl)

In [0]:
vw_name_saida = f"{environment_tbl}vw_gold_modelo_{id_projeto}"
print(vw_name_saida)

In [0]:
root_remote_path = f"/mnt/trusted/datalake/ia/projetos/{id_projeto}/"
print(root_remote_path)

In [0]:
root_remote_config_path = f"{root_remote_path}config/{environment}/"
print(root_remote_config_path)

In [0]:
root_remote_data_path = f"{root_remote_path}data/{environment}/envio/"
print(root_remote_data_path)

In [0]:
year, month, day = data_execucao_modelo.split("-")
remote_path = f"{root_remote_data_path}{year}/{month}/"
print(remote_path)

In [0]:
# IDADE_LIMITE = 75

# Import libs

In [0]:
import os
import pandas as pd
import openpyxl
import copy
import json
import subprocess

from datetime import date
from pathlib import Path

In [0]:
# shell_command = "pwd"
# output = subprocess.check_output(shell_command, shell=True)
# current_folder = os.path.join(output.decode().strip(), id_projeto) + "/"

current_folder = os.path.join("/tmp", id_projeto) + "/"

Path(current_folder).mkdir(parents=True, exist_ok=True)

print(current_folder)

# Configuração de display

In [0]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', 230)

# Exporta os dados para Excel

In [0]:
vw_name_saida

In [0]:
# %sql
# describe ia.dev_vw_gold_modelo_endometriose

In [0]:
def get_data(unidades):
  df = spark.sql(f"""
      select
          dataExecucaoModelo,
          idPredicao,
          idExame,
          date_format(date(dataExame), 'dd/MM/yyyy') as dataExame,
          date_format(date(dataLiberacaoLaudo), 'dd/MM/yyyy') as dataLiberacaoLaudo,
          nomeConvenio,
          -- tipoCodigo,
          codigo,
          nomeCodigo,
          descricaoProcedimento,
          -- idUnidade,
          nomeUnidade,
          -- tipoUnidade,
          -- idMedico,
          ufCrm,
          cnpjUnidade,
          regionalUnidade,
          numCrm,
          if (nomeMedico != 'None', nomeMedico, '') as nomeMedico,
          -- idPaciente,
          cpfPaciente,
          nomePaciente,
          -- dataNascimentoPaciente,
          idadePaciente,
          sexoPaciente,
          telefoneContato,
          laudoOriginal,
          -- laudoDuplicado,
          -- dataCarga,
          -- laudoOriginalLimpo,
          -- laudoTratado,
          -- listaSintomas,
          -- listaAchados,
          classificacao,
          -- laudoResumido,
          -- classificacaoEndometriose,
          "" as achado,
          "" as observacoes,
          "" as destino

      from ia.{vw_name_saida}
      
      where dataExecucaoModelo = '{data_execucao_modelo}'
        and classificacaoEndometriose = 'Positivo'
        and (
          idUnidade in ({unidades})
          or
          "{unidades}" = "'todas'"
        )
        
      order by
          regionalUnidade,
          nomeUnidade,
          nomePaciente
  """).toPandas()

  return df

In [0]:
def get_cols():
    """Dicionário de/para com os nomes das colunas de saída"""
    dic_col_names = {
        "dataExecucaoModelo": "dataExecucaoModelo",
        "idPredicao": "idPredicao",
        "idExame": "idExame",

        "nomePaciente": "Nome Paciente",
        "cpfPaciente": "CPF Paciente",
        "idadePaciente": "Idade Paciente",
        "sexoPaciente": "Sexo Paciente",
        "telefoneContato": "Telefone Contato",

        "nomeUnidade": "Unidade",
        "cnpjUnidade": "CNPJ Unidade",
        "regionalUnidade": "Regional Unidade",
        # "tipoUnidade": "Tipo Unidade",

        "numCrm": "CRM",
        "ufCrm": "UF CRM",
        "nomeMedico": "Nome Médico",

        "dataExame": "Data Exame",
        "dataLiberacaoLaudo": "Data Liberação Laudo",
        "nomeConvenio": "Convênio",
        "codigo": "Código TUSS",
        "nomeCodigo": "Descrição TUSS",
        "descricaoProcedimento": "Nome Procedimento",
        "laudoOriginal": "Laudo",
        "classificacao": "Classificação",
        # "score": "Score",
        "achado": "Achado",
        "observacoes": "Observações",
        "destino": "Destino"
    }

    return dic_col_names

In [0]:
def export_to_excel(df, dic_col_names, unidade):
    file_name = f"{id_projeto}_{unidade}_{data_execucao_modelo}.xlsx"
    local_file = f"{current_folder}{file_name}"
    remote_file = f"{remote_path}{file_name}"

    cols = [col for col in dic_col_names.keys()]
    df[cols].to_excel(local_file, index=False, freeze_panes=(1, 4), engine='xlsxwriter')

    return {
        "unidade": unidade,
        "nomeArquivo": file_name,
        "arquivoLocal": local_file,
        "arquivoRemoto": remote_file,
        "caminhoRemoto": remote_path.replace("/mnt", ""),
        "registros": len(df.index),
        "dataProcessamento": data_execucao_modelo,
        "dataProcessamentoFormatada": datetime.strptime(data_execucao_modelo, "%Y-%m-%d").strftime("%d/%m/%Y"),
    }

In [0]:
def get_cols_config():
    """Define tamanho fixo para algumas colunas. ( -1 = Coluna oculta )"""
    dic_col_fixed_width = {
        "dataExecucaoModelo": -1,
        "idPredicao": -1,
        "idExame": -1,
        "Laudo": 150,
        "Achado": 30,
        "Observações": 30,
    }
    return dic_col_fixed_width

In [0]:
# def format_excel(full_file_name, dic_col_names, dic_col_fixed_width):
#     import math
#     from openpyxl.styles import Alignment, Font, PatternFill
#     from openpyxl.worksheet.datavalidation import DataValidation

#     wb = openpyxl.load_workbook(full_file_name)
#     sheet = wb.active

#     dic_col_size = {}

#     dv_achado = DataValidation(type="list", formula1='"1 - Alta probabilidade,2 - Média probabilidade,3 - Baixa probabilidade,4 - Descartada probabilidade"', allow_blank=True)
#     dv_achado.showInputMessage = True
#     dv_achado.showErrorMessage = True
#     dv_achado.error = "Este valor não corresponde às restrições de valições de dados definidas para esta célula."
#     dv_achado.errorTitle = 'Valor Inválido'

#     # dv_linha_cuidado = DataValidation(type="list", formula1='"1 - Cirrose,2 - Esteatose,3 - Transplante"', allow_blank=True)
#     # dv_linha_cuidado.showInputMessage = True
#     # dv_linha_cuidado.showErrorMessage = True
#     # dv_linha_cuidado.error = "Este valor não corresponde às restrições de valições de dados definidas para esta célula."
#     # dv_linha_cuidado.errorTitle = 'Valor Inválido'

#     # col_idade = ""

#     wrap_text = [
#         "Laudo",
#         # "Tipo Exame",
#         # "Nome Hospital",
#     ]

#     h_align_center = [
#         "CPF Paciente",
#         "Idade Paciente",
#         "Sexo Paciente",
#         "Telefone Contato",
#         "CNPJ Unidade",
#         "Regional Unidade"
#         "CRM",
#         "UF CRM",
#         "Data Liberação Laudo",
#         "Código TUSS",
#         "Data Exame",
#         "Regional Unidade",
#     ]

#     for cols in sheet.iter_cols():
#         for cell in cols:
#             # Renomeia as colunas
#             if cell.row == 1:
#                 cell.value = dic_col_names.get(cell.value, cell.value)

#             column_letter = cell.column_letter
#             len_value = len(str(cell.value))
            
#             if len_value > dic_col_size.get(column_letter, 0):
#                 dic_col_size[column_letter] = len_value

#             if cols[0].value == "Achado" and cell.row > 1:
#                 dv_achado.add(cell)

#             # if cols[0].value == "Linha de Cuidado" and cell.row > 1:
#             #     dv_linha_cuidado.add(cell)

#             # if cols[0].value == "Idade Paciente":
#             #     if cell.row > 1:
#             #         idade = cell.value
#             #         if idade > IDADE_LIMITE:
#             #             cell.fill = PatternFill(fill_type="solid", start_color="F2DCDB")
#             #             cell.font = Font(color="9C0006")

#             alignment = copy.copy(cell.alignment)
#             alignment.vertical = "center"

#             if cell.row > 1 and cols[0].value in wrap_text:
#                 alignment.wrapText = True

#             if cell.row > 1 and cols[0].value in h_align_center:
#                 alignment.horizontal = "center"
        
#             cell.alignment = alignment

#     # Ajusta a largura das colunas
#     for k, v in dic_col_size.items():
#         adjusted_width = dic_col_fixed_width.get(sheet[f"{k}1"].value)

#         if adjusted_width is None:
#             adjusted_width = int((v + 2))

#         if adjusted_width == -1:
#             sheet.column_dimensions[k].hidden = True
#         else:
#             sheet.column_dimensions[k].width = adjusted_width
                
#     sheet.add_data_validation(dv_achado)
#     # sheet.add_data_validation(dv_linha_cuidado)

#     wb.save(full_file_name)
#     wb.close()


In [0]:
def format_excel(full_file_name, dic_col_names, dic_col_fixed_width):
    import math
    from openpyxl.styles import Alignment, Font, PatternFill
    from openpyxl.worksheet.datavalidation import DataValidation

    wb = openpyxl.load_workbook(full_file_name)
    sheet = wb.active

    dic_col_size = {}

    dv_achado = DataValidation(type="list", formula1='"1 - Alta probabilidade,2 - Média probabilidade,3 - Baixa probabilidade,4 - Descartada probabilidade"', allow_blank=True)
    dv_achado.showInputMessage = True
    dv_achado.showErrorMessage = True
    dv_achado.error = "Este valor não corresponde às restrições de valições de dados definidas para esta célula."
    dv_achado.errorTitle = 'Valor Inválido'

    dv_destino = DataValidation(type="list", formula1='"1 - Navegação,2 - Não"', allow_blank=True)
    dv_destino.showInputMessage = True
    dv_destino.showErrorMessage = True
    dv_destino.error = "Este valor não corresponde às restrições de valições de dados definidas para esta célula."
    dv_destino.errorTitle = 'Valor Inválido'

    # col_idade = ""

    wrap_text = [
        "Laudo",
        # "Tipo Exame",
        # "Nome Hospital",
    ]

    h_align_center = [
        "CPF Paciente",
        "Idade Paciente",
        "Sexo Paciente",
        "Telefone Contato",
        "CNPJ Unidade",
        "Regional Unidade"
        "CRM",
        "UF CRM",
        "Data Liberação Laudo",
        "Código TUSS",
        "Data Exame",
        "Regional Unidade",
    ]

    for cols in sheet.iter_cols():
        for cell in cols:
            # Renomeia as colunas
            if cell.row == 1:
                cell.value = dic_col_names.get(cell.value, cell.value)

            column_letter = cell.column_letter
            len_value = len(str(cell.value))
            
            if len_value > dic_col_size.get(column_letter, 0):
                dic_col_size[column_letter] = len_value

            if cols[0].value == "Achado" and cell.row > 1:
                dv_achado.add(cell)

            if cols[0].value == "Destino" and cell.row > 1:
                dv_destino.add(cell)

            # if cols[0].value == "Idade Paciente":
            #     if cell.row > 1:
            #         idade = cell.value
            #         if idade > IDADE_LIMITE:
            #             cell.fill = PatternFill(fill_type="solid", start_color="F2DCDB")
            #             cell.font = Font(color="9C0006")

            alignment = copy.copy(cell.alignment)
            alignment.vertical = "center"

            if cell.row > 1 and cols[0].value in wrap_text:
                alignment.wrapText = True

            if cell.row > 1 and cols[0].value in h_align_center:
                alignment.horizontal = "center"
        
            cell.alignment = alignment

    # Ajusta a largura das colunas
    for k, v in dic_col_size.items():
        adjusted_width = dic_col_fixed_width.get(sheet[f"{k}1"].value)

        if adjusted_width is None:
            adjusted_width = int((v + 2))

        if adjusted_width == -1:
            sheet.column_dimensions[k].hidden = True
        else:
            sheet.column_dimensions[k].width = adjusted_width
                
    sheet.add_data_validation(dv_achado)
    sheet.add_data_validation(dv_destino)

    wb.save(full_file_name)
    wb.close()


In [0]:
def copy_files(source, target):
    source = f"file://{source}"
    # print("-"*120)
    print("Copiando arquivo")
    print(f"De  : {source}")
    print(f"Para: {target}")

    dbutils.fs.cp(source, target, recurse=True)

In [0]:
def extract_data(unidade, ids):
    print("-" * 120)
    print(f"Extraindo dados para a unidade: {unidade}")
    
    _df_export = get_data(ids)

    print(f"Registros encontrados: {len(_df_export.index)}")

    _dic_col_names = get_cols()
    
    _info = export_to_excel(_df_export, _dic_col_names, unidade)
    
    _dic_col_fixed_width = get_cols_config()
    
    format_excel(_info['arquivoLocal'], _dic_col_names, _dic_col_fixed_width)
    
    copy_files(_info['arquivoLocal'], _info['arquivoRemoto'])

    return _info

In [0]:
def clean(folder):
    files = os.listdir(folder)
    for file in files:
        file_path = os.path.join(folder, file)        
        if os.path.isfile(file_path):
            print(f"Excluido arquivo: {file_path}")
            os.remove(file_path)

In [0]:
df = spark.read.option("multiline","true").json(f"{root_remote_config_path}unidades.json")
df.display()

In [0]:
files = []

for row in df.collect():
    if row.unidades is None:
        print("-" * 120)
        print(f"Unidade: {row.unidade} - Não configurada!")
        continue

    info = extract_data(row.unidade, row.unidades)
    files.append(info)

In [0]:
metadados = f"{current_folder}{id_projeto}_metadados_envio.json"

json_data = json.dumps(files)

with open(metadados, "w") as file:
    file.write(json_data)

In [0]:
copy_files(metadados, root_remote_data_path)

In [0]:
clean(current_folder)

In [0]:
!ls -lha {current_folder}

In [0]:
!rm -rf {current_folder}

In [0]:
dbutils.notebook.exit("Fim do processamento!")